# Baseline Grokking experiment


In [ ]:
import random
import numpy as np
import torch as t
import torch.optim as optim
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from model import Config, Transformer

try:
    from helpers import gen_train_test, cross_entropy_high_precision
except Exception:
    # Small local fallbacks matching helpers.py. This keeps the notebook runnable
    # even when optional helper imports like wandb are not installed.
    def gen_train_test(config):
        pairs = [(i, j, config.p) for i in range(config.p) for j in range(config.p)]
        random.seed(config.seed)
        random.shuffle(pairs)
        div = int(config.frac_train * len(pairs))
        return pairs[:div], pairs[div:]

    def cross_entropy_high_precision(logits, labels):
        logprobs = t.nn.functional.log_softmax(logits.to(t.float32), dim=-1)
        return -t.gather(logprobs, index=labels[:, None], dim=-1).mean()

t.manual_seed(0)
np.random.seed(0)
random.seed(0)

## Setup
the notebook only defines the small helpers needed for labels, evaluation, and plotting.

In [ ]:
def labels_for(data, p, device):
    return ((data[:, 0] + data[:, 1]) % p).to(device)

@t.no_grad()
def evaluate(model, data, p, device):
    labels = labels_for(data, p, device)
    logits = model(data)[:, -1, :p]
    loss = cross_entropy_high_precision(logits, labels)
    acc = (logits.argmax(dim=-1) == labels).float().mean()
    return loss.item(), acc.item()

def plot_curves(history):
    colors = {"train": "#636EFA", "test": "#EF553B"}
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss", "Accuracy"))
    for key, name in [("train_loss", "train"), ("test_loss", "test")]:
        fig.add_trace(go.Scatter(x=history["epoch"], y=history[key], mode="lines", name=name, line=dict(color=colors[name])), row=1, col=1)
    for key, name in [("train_acc", "train"), ("test_acc", "test")]:
        fig.add_trace(go.Scatter(x=history["epoch"], y=history[key], mode="lines", name=name, line=dict(color=colors[name]), showlegend=False), row=1, col=2)
    fig.update_yaxes(type="log", title_text="cross entropy", row=1, col=1)
    fig.update_yaxes(range=[0, 1.02], title_text="accuracy", row=1, col=2)
    fig.update_xaxes(title_text="epoch")
    fig.update_layout(title="Minimal grokking run", template="plotly_white", width=950, height=420)
    fig.show()

@t.no_grad()
def plot_final_grid(model, all_data, train_data, config):
    logits = model(all_data)[:, -1, :config.p]
    preds = logits.argmax(dim=-1).detach().cpu().reshape(config.p, config.p).numpy()
    labels = labels_for(all_data, config.p, config.device).detach().cpu().reshape(config.p, config.p).numpy()
    train_mask = np.zeros((config.p, config.p), dtype=float)
    for x, y, _ in train_data:
        train_mask[x, y] = 1.0

    fig = make_subplots(rows=1, cols=3, subplot_titles=("prediction", "wrong cells", "train split"))
    fig.add_trace(go.Heatmap(z=preds, colorscale="Viridis", showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=(preds != labels).astype(float), colorscale="Reds", showscale=False), row=1, col=2)
    fig.add_trace(go.Heatmap(z=train_mask, colorscale="Greys", showscale=False), row=1, col=3)
    fig.update_xaxes(title_text="y")
    fig.update_yaxes(title_text="x", autorange="reversed")
    fig.update_layout(title="Learned modular addition table", template="plotly_white", width=950, height=360)
    fig.show()

## Train

The defaults below are intentionally small. Increase `p`, `d_model`, or `num_epochs` if you want a cleaner full grokking phase transition.

In [ ]:
config = Config(
    p=113,
    d_vocab=114,
    d_model=128,
    d_mlp=512,
    num_heads=4,
    frac_train=0.3,
    lr=1e-3,
    weight_decay=1.0,
    num_epochs=40000,
    seed=0,
)
assert config.d_vocab == config.p + 1, "d_vocab must include p numeric tokens plus the special '=' token"

"""config = Config( #! Intéressant, ne grok pas avec ces params
    p=31,
    d_vocab=32,
    d_model=64, 
    d_mlp=256,
    num_heads=4,
    frac_train=0.3,
    lr=1e-3,
    weight_decay=1.0,
    num_epochs=13000,
    seed=0,
)
"""

train, test = gen_train_test(config)
train_data = t.tensor(train, dtype=t.long, device=config.device)
test_data = t.tensor(test, dtype=t.long, device=config.device)
all_data = t.tensor([(i, j, config.p) for i in range(config.p) for j in range(config.p)], dtype=t.long, device=config.device)

model = Transformer(config, use_cache=False).to(config.device)
optimizer = optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay, betas=(0.9, 0.98))
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(step / 10, 1.0))

history = {"epoch": [], "train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
eval_every = 50

for epoch in range(config.num_epochs + 1):
    if epoch % eval_every == 0:
        model.eval()
        train_loss, train_acc = evaluate(model, train_data, config.p, config.device)
        test_loss, test_acc = evaluate(model, test_data, config.p, config.device)
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        if epoch % 500 == 0:
            print(f"epoch {epoch:5d} | train loss {train_loss:.4f} acc {train_acc:.3f} | test loss {test_loss:.4f} acc {test_acc:.3f}")

    if epoch == config.num_epochs:
        break

    model.train()
    logits = model(train_data)[:, -1, :config.p]
    labels = labels_for(train_data, config.p, config.device)
    loss = cross_entropy_high_precision(logits, labels)
    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

print(f"final train acc {history['train_acc'][-1]:.3f}, final test acc {history['test_acc'][-1]:.3f}")
plot_curves(history)

## Inspect the learned operation

In [ ]:
plot_final_grid(model, all_data, train, config)